#Changing HTML to String

In [1]:
!pip install ftfy
import pandas as pd
import html
from google.colab import files
import ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00


In [2]:
files.upload()
import pandas as pd

df = pd.read_csv("all_tasks_90_sub_23_12.csv", encoding="utf-8", encoding_errors="replace")
df = df.applymap(
    lambda x: ftfy.fix_text(x) if isinstance(x, str) else x
)
df.head()


Saving all_tasks_90_sub_23_12.csv to all_tasks_90_sub_23_12.csv


/tmp/ipykernel_912/2828561421.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(


,msg_id,gpt_interface_id,participant_id,unix_time_when_msg_sent_to_gpt,date_time_when_msg_sent_to_gpt,unix_time_when_msg_received_from_gpt,date_time_when_msg_received_from_gpt,message_to_gpt,message_from_gpt,msg_count_within_p,TASK,subject_id,subject_group,Sex,Age,response_time_sec,response_time_min,words_in_message_to_gpt
0,39,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,Where is best to stay in andora,"Andorra, a small principality nestled between ...",1,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,7.9523,0.132538,7
1,40,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,What activities is there to do on a low budget,There are plenty of activities you can enjoy o...,2,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,10.7440,0.179067,10
2,41,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,Where is best to eat in Andora,Andorra offers a diverse dining scene that cat...,3,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,16.4215,0.273692,7
3,42,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,Is there beaches to visit in andora,Andorra is a landlocked country located in the...,4,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,8.7013,0.145022,7
4,43,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,What is the temperature like in andorra,Andorra's climate is characterized by its moun...,5,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,13.2633,0.221055,7


#Age Masking

logic is like this:
1) number in words (example: "thirty five")
2) patterns
3) birthday contex

according to 1+2+3 we can mask a string
loging section is for debugging and works only locally (not here in colab)




WHAT YOU NEED TO DO:
1) upload a file called "data.csv"
2) run the following cell

In [3]:
import re

# -----------------------
# Number patterns
# -----------------------
NUMBER_WORDS = (
    r"(?:one|two|three|four|five|six|seven|eight|nine|ten|"
    r"eleven|twelve|thirteen|fourteen|fifteen|sixteen|"
    r"seventeen|eighteen|nineteen|twenty|thirty|forty|"
    r"fifty|sixty|seventy|eighty|ninety|hundred)"
)

NUMBER_PATTERN = fr"(?:\d+|{NUMBER_WORDS}(?:[- ]{NUMBER_WORDS})?)"

# -----------------------
# Strong patterns (always applied)
# group_idx can be int or tuple of ints
# -----------------------
PATTERNS = [

    # X years old / X-year-old / typos
    (re.compile(
        fr"\b({NUMBER_PATTERN})(\s*[- ]?(?:years?|yrs?|tear|yeat)[-\s]+old)\b",
        re.IGNORECASE
    ), 1),

    # X yo / y.o.
    (re.compile(
        fr"\b({NUMBER_PATTERN})(\s*[- ]?(?:y\.?o\.?|y/o))\b",
        re.IGNORECASE
    ), 1),

    # aged X
    (re.compile(
        fr"\b(aged?\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),

    # aged X to Y / aged X-Y
    (re.compile(
        fr"\b(aged?\s+)({NUMBER_PATTERN})\s*(?:to|-|–)\s*({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), (2, 3)),

    # turning X
    (re.compile(
        fr"\b(turning\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),

    # Xth birthday
    (re.compile(
        fr"\b({NUMBER_PATTERN})(?:st|nd|rd|th)?(\s+birt?h?day)\b",
        re.IGNORECASE
    ), 1),

    # will be X (filtered later by context)
    (re.compile(
        fr"\b(will\s+be\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),

    # in his/her/their/my 20s
    (re.compile(
        fr"\b(in\s+(?:her|his|their|my|our)\s+)({NUMBER_PATTERN})('?s)\b",
        re.IGNORECASE
    ), 2),

    # in their early/mid/late 20s
    (re.compile(
        fr"\b(in\s+their\s+(?:early|mid|late)\s+)({NUMBER_PATTERN})('?s)\b",
        re.IGNORECASE
    ), 2),
]

# -----------------------
# Weak patterns (require age context)
# -----------------------
WEAK_PATTERNS = [

    # is/was X
    (re.compile(
        fr"\b((?:is|was)\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),

    # I am / I'm X
    (re.compile(
        fr"\b((?:I\s+am|I'm)\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),
]

# -----------------------
# Masking function
# -----------------------
def mask_text(text: str, mask_str: str = "###") -> str:
    if not text:
        return text

    masked_text = text

    # Context required for weak patterns
    has_context = re.search(
        r"\b(birt?h?day|bday|card|greeting|turning|years?\s+old|aged?|age|"
        r"son|daughter|child|grandson|granddaughter)\b",
        text,
        re.IGNORECASE
    )

    active_patterns = PATTERNS + (WEAK_PATTERNS if has_context else [])

    for pattern, group_idx in active_patterns:

        def replacer(match):
            full = match.group(0)

            # support single group or multiple groups
            group_idxs = (
                group_idx if isinstance(group_idx, tuple) else (group_idx,)
            )

            # replace from right to left to keep indices valid
            for gi in sorted(group_idxs, reverse=True):
                g_start = match.start(gi) - match.start(0)
                g_end = match.end(gi) - match.start(0)
                full = full[:g_start] + mask_str + full[g_end:]

            return full

        masked_text = pattern.sub(replacer, masked_text)

    return masked_text



In [4]:
for col in ['message_to_gpt', 'message_from_gpt']:
    if col in df.columns:
        df[f'{col}_age_masked'] = df[col].apply(
            lambda x: mask_text(x) if isinstance(x, str) else x
        )

In [5]:
df['age_masked_to_gpt'] = (
    df['message_to_gpt'] != df['message_to_gpt_age_masked']
)

df['age_masked_from_gpt'] = (
    df['message_from_gpt'] != df['message_from_gpt_age_masked']
)

cols_to_export = [
    'participant_id',
    'message_to_gpt_age_masked',
    'message_from_gpt_age_masked'
]

df_export = df[cols_to_export]
df_export.to_csv("masked_messages_only.csv", index=False)

In [6]:
df['age_masked_from_gpt'].sum()
df['age_masked_to_gpt'].sum()

np.int64(42)

In [7]:
cols_to_keep = [
    'unix_time_when_msg_sent_to_gpt',
    'unix_time_when_msg_received_from_gpt',
    'msg_count_within_p',
    'TASK',
    'subject_id',
    'message_to_gpt_age_masked',
    'message_from_gpt_age_masked',
    'subject_group',
    'Sex',
    'Age']


In [8]:
df_greeting = df[df['TASK'] == 'greeting_task'].copy()
df_trip = df[df['TASK'] == 'trip_task'].copy()
df_greeting = df_greeting[cols_to_keep].copy()
df_trip = df_trip[cols_to_keep].copy()

In [9]:
df_trip = (
    df_trip
    .sort_values(
        by=['subject_id', 'unix_time_when_msg_sent_to_gpt']
    )
    .assign(
        turn_index=lambda x: x.groupby('subject_id').cumcount() + 1
    )
)


In [10]:
df_greeting = (
    df_greeting
    .sort_values(
        by=['subject_id', 'unix_time_when_msg_sent_to_gpt']
    )
    .assign(
        turn_index=lambda x: x.groupby('subject_id').cumcount() + 1
    )
)


In [11]:
df_trip

,unix_time_when_msg_sent_to_gpt,unix_time_when_msg_received_from_gpt,msg_count_within_p,TASK,subject_id,message_to_gpt_age_masked,message_from_gpt_age_masked,subject_group,Sex,Age,turn_index
186,1.761598e+09,1.761598e+09,1,trip_task,568a9909316b10000c50ae46,I would like to spend a four day holiday in An...,Andorra is a small but beautiful country nestl...,Older_Adults,Female,75,1
187,1.761598e+09,1.761598e+09,2,trip_task,568a9909316b10000c50ae46,I am in my seventies. I would like to hike but...,Certainly! Here's a revised itinerary for your...,Older_Adults,Female,75,2
277,1.762889e+09,1.762889e+09,1,trip_task,56b12c0e9f1826000cbcc76c,I want you to help me plan a 4 day trip to And...,Planning a budget-friendly trip from the UK to...,Young_Adults,Female,37,1
278,1.762890e+09,1.762890e+09,2,trip_task,56b12c0e9f1826000cbcc76c,thank you. What activities are there to do in ...,Andorra offers a variety of activities for vis...,Young_Adults,Female,37,2
279,1.762890e+09,1.762890e+09,3,trip_task,56b12c0e9f1826000cbcc76c,does Andorra have any interesting history?,"Yes, Andorra has a fascinating history, shaped...",Young_Adults,Female,37,3
...,...,...,...,...,...,...,...,...,...,...,...
41,1.759848e+09,1.759848e+09,2,trip_task,6783a1eee721ad7b19a43435,"I am not too keen on outdoor activities , can ...","Of course, I can adjust the itinerary to focus...",Older_Adults,Female,66,2
42,1.759848e+09,1.759848e+09,3,trip_task,6783a1eee721ad7b19a43435,can you tell me more about the best local rest...,Certainly! Here is some information on the bes...,Older_Adults,Female,66,3
43,1.759848e+09,1.759848e+09,4,trip_task,6783a1eee721ad7b19a43435,I always like to take gifts back home for frie...,When considering gifts to bring back from Ando...,Older_Adults,Female,66,4
44,1.759848e+09,1.759848e+09,5,trip_task,6783a1eee721ad7b19a43435,Would you recommend the spa products,"Yes, I would recommend considering the spa pro...",Older_Adults,Female,66,5


In [12]:
df_trip['order_mismatch'] = (
    df_trip['turn_index'] != df_trip['msg_count_within_p']
)
df_trip['order_mismatch'].value_counts()


,count
order_mismatch,
False,435
True,9


In [13]:
df_trip.loc[
    df_trip['order_mismatch'],
    ['subject_id',
     'unix_time_when_msg_sent_to_gpt',
     'msg_count_within_p',
     'turn_index',
     'message_to_gpt_age_masked',
     'subject_group',
     'Age']
].sort_values(['subject_id', 'turn_index'])


,subject_id,unix_time_when_msg_sent_to_gpt,msg_count_within_p,turn_index,message_to_gpt_age_masked,subject_group,Age
299,5827634ea80bf4000199994b,1.762897e+09,1,2,"plan a trip to Andorra for me. Four days, thre...",Older_Adults,68
431,5ec1cb44516ced000af5f279,1.765211e+09,1,2,I am travelling to Andorra in September for a ...,Young_Adults,30
162,5f836c9938d2da4fb75631ef,1.761592e+09,3,4,what about passports and customs controls? Am ...,Older_Adults,69
183,649d81f47df8018a01c555dd,1.761596e+09,2,3,Thanks. We want to stop somewhere quite quiet ...,Older_Adults,70
432,6674a611c78207e9009eaadd,1.765211e+09,1,2,planning a trip to benidorm,Young_Adults,35
201,66a90e7caa371f25183bdce2,1.761680e+09,5,6,WHich reviews do you look at?,Older_Adults,72
49,673dbbee9983e0d8084c8a76,1.759849e+09,2,3,I am only going for 4-days. How should I fill ...,Older_Adults,69
50,673dbbee9983e0d8084c8a76,1.759849e+09,2,4,I am only going for 4-days. How should I fill ...,Older_Adults,69
51,673dbbee9983e0d8084c8a76,1.759849e+09,2,5,I am only going for 4-days. How should I fill ...,Older_Adults,69


In [14]:
df_greeting['order_mismatch'] = (
    df_greeting['turn_index'] != df_greeting['msg_count_within_p']
)

df_greeting['order_mismatch'].value_counts()


,count
order_mismatch,
False,450
True,5


In [15]:
df_greeting.loc[
    df_greeting['order_mismatch'],
    ['subject_id',
     'unix_time_when_msg_sent_to_gpt',
     'msg_count_within_p',
     'turn_index',
     'message_to_gpt_age_masked',
     'subject_group',
     'Age']
].sort_values(['subject_id', 'turn_index'])


,subject_id,unix_time_when_msg_sent_to_gpt,msg_count_within_p,turn_index,message_to_gpt_age_masked,subject_group,Age
985,66e4906547365a7208341b04,1.761593e+09,1,2,Can you create a birthday message for my husba...,Young_Adults,34
984,66e4906547365a7208341b04,1.761593e+09,1,3,Can you create a birthday message for my husba...,Young_Adults,34
987,66e4906547365a7208341b04,1.761593e+09,1,4,Can you create a birthday message for my husba...,Young_Adults,34
1238,671f5208a60be0a55c28af8c,1.765125e+09,5,6,We live near the beach on Hayling Island. What...,Young_Adults,38
1240,671f5208a60be0a55c28af8c,1.765125e+09,6,7,What is a good poem that reflects my husband's...,Young_Adults,38


In [16]:
# ------------------------------------------------
# Helper function: build a full conversation
# for a single subject
# ------------------------------------------------
def build_full_conversation(df_subject: pd.DataFrame) -> str:
    """
    Build a full conversation transcript for one subject.
    Each turn is formatted as:
        user: ...
        assistant: ...
    Turns are concatenated in chronological order.
    """

    conversation_turns = []

    for _, row in df_subject.iterrows():
        user_msg = row['message_to_gpt_age_masked']
        assistant_msg = row['message_from_gpt_age_masked']

        if isinstance(user_msg, str) and user_msg.strip():
            conversation_turns.append(f"[user] {user_msg}")

        if isinstance(assistant_msg, str) and assistant_msg.strip():
            conversation_turns.append(f"[assistant] {assistant_msg}")

    return "\n\n".join(conversation_turns)


# ------------------------------------------------
# Helper function: build conversation-level DF
# from a turn-level DF (trip or greeting)
# ------------------------------------------------
def build_conversation_df(df_task: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a turn-level dataframe (one row per message)
    and returns a conversation-level dataframe
    (one row per subject).
    """

    # Ensure correct order
    df_task = (
        df_task
        .sort_values(['subject_id', 'turn_index', 'subject_group','Age'])
        .reset_index(drop=True)
    )

    # Build conversations
    df_conversations = (
        df_task
        .groupby('subject_id', group_keys=False)
        .apply(build_full_conversation)
        .reset_index(name='full_conversation')
    )

    # Attach task label (one per subject)
    task_per_subject = (
        df_task[['subject_id', 'TASK', 'subject_group','Age']]
        .drop_duplicates(subset='subject_id')
    )

    df_conversations = df_conversations.merge(
        task_per_subject,
        on='subject_id',
        how='left'
    )

    return df_conversations


# ------------------------------------------------
# Build conversation-level dataframes
# ------------------------------------------------

df_conversations_trip = build_conversation_df(df_trip)
df_conversations_greeting = build_conversation_df(df_greeting)


# ------------------------------------------------
# Quick sanity checks
# ------------------------------------------------
print("Trip conversations:", df_conversations_trip.shape[0])
print("Greeting conversations:", df_conversations_greeting.shape[0])

print("\nExample Trip conversation:\n")
print(df_conversations_trip['full_conversation'].iloc[0][:1000])

#print("\nExample Greeting conversation:\n")
#print(df_conversations_greeting['full_conversation'].iloc[0][:1000])


Trip conversations: 90
Greeting conversations: 90

Example Trip conversation:

[user] I would like to spend a four day holiday in Andorra travelling on my own. I like staying in hostels, preferably YHA hostels. Can you suggest where I can stay and the best places to visit in Andorra. Thank you

[assistant] Andorra is a small but beautiful country nestled in the Pyrenees between France and Spain. It is known for its ski resorts, hiking trails, and tax-free shopping. Here's a guide to help you plan your four-day solo trip to Andorra, focusing on hostel stays and key attractions.

1. **Accommodation:**
   - While Andorra doesn't have YHA hostels specifically, it does offer various budget-friendly accommodations that cater to solo travelers.
   - Consider staying at Alberg la Comella, a hostel that provides comfortable lodging and is conveniently located for accessing Andorra la Vella, the capital city.
   - Another option is Barri Antic Hostel & Pub, situated in the heart of Andorra la Ve

/tmp/ipykernel_912/1283483916.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_full_conversation)
/tmp/ipykernel_912/1283483916.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_full_conversation)


In [17]:
df_conversations_trip

,subject_id,full_conversation,TASK,subject_group,Age
0,568a9909316b10000c50ae46,[user] I would like to spend a four day holida...,trip_task,Older_Adults,75
1,56b12c0e9f1826000cbcc76c,[user] I want you to help me plan a 4 day trip...,trip_task,Young_Adults,37
2,56f699e876348f000c883bba,[user] Please can you help me plan a trip to A...,trip_task,Young_Adults,30
3,57698d52688a5c00018d958d,[user] I am trying to plan a 4 day trip to And...,trip_task,Older_Adults,65
4,578bb26f6cc44500010448c8,[user] I'd like some help in planning an itine...,trip_task,Older_Adults,70
...,...,...,...,...,...
85,6725c4a8e3ab0787551acf38,[user] i want to plan a trip for 2 adults to a...,trip_task,Older_Adults,65
86,673dbbee9983e0d8084c8a76,"[user] what are the ""must-see"" sights in Andor...",trip_task,Older_Adults,69
87,6764260cc5b6436f25820b52,[user] I am going on a 4 day trip to Andorra. ...,trip_task,Young_Adults,31
88,6772e76feabb30f82fcd8f59,[user] I need to make an itinerary for a 4-day...,trip_task,Young_Adults,38


In [18]:
df_conversations_trip['word_count'] = (
    df_conversations_trip['full_conversation']
    .str.split()
    .str.len()
)
df_conversations_trip['word_count'].describe()

,word_count
count,90.000000
mean,1682.922222
std,645.583444
min,476.000000
25%,1245.000000
50%,1558.000000
75%,2206.000000
max,3217.000000


In [19]:
df_conversations_greeting['word_count'] = (
    df_conversations_greeting['full_conversation']
    .str.split()
    .str.len()
)
df_conversations_greeting['word_count'].describe()

,word_count
count,90.000000
mean,649.833333
std,376.036403
min,150.000000
25%,356.250000
50%,575.500000
75%,828.500000
max,2261.000000


In [20]:
df_conversations_trip['approx_tokens'] = (df_conversations_trip['word_count'] * 1.3).astype(int)

(df_conversations_trip['approx_tokens'] > 4096).sum()

np.int64(2)

In [21]:
df_conversations_trip.sort_values('approx_tokens', ascending=False).head(10)
df_conversations_trip.sort_values('approx_tokens', ascending=False)[
    ['subject_id', 'word_count', 'approx_tokens']
].head(10)

,subject_id,word_count,approx_tokens
88,6772e76feabb30f82fcd8f59,3217,4182
73,665f3e69f83003c1df913675,3176,4128
58,6144d8d264b28a29e10ad3de,3055,3971
86,673dbbee9983e0d8084c8a76,2975,3867
84,672107688fa36a5213347a7a,2845,3698
83,671f5208a60be0a55c28af8c,2677,3480
10,59d949cf8fd07a000184247e,2577,3350
12,5ab69c4ba3ba7a0001b405a6,2571,3342
3,57698d52688a5c00018d958d,2539,3300
60,62e15cc80e4bd2ad93f10f5e,2531,3290


In [22]:
import pandas as pd
from transformers import AutoTokenizer

# tokenizers
tokenizer_longformer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")
tokenizer_roberta = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

def count_tokens(text, tokenizer):
    if pd.isna(text):
        return 0
    return len(tokenizer.encode(str(text), add_special_tokens=True, truncation=False))

# exact token counts
df_conversations_trip['tokens_longformer'] = df_conversations_trip['full_conversation'].apply(
    lambda x: count_tokens(x, tokenizer_longformer)
)

df_conversations_trip['tokens_roberta'] = df_conversations_trip['full_conversation'].apply(
    lambda x: count_tokens(x, tokenizer_roberta)
)

df_conversations_greeting['tokens_longformer'] = df_conversations_greeting['full_conversation'].apply(
    lambda x: count_tokens(x, tokenizer_longformer)
)

df_conversations_greeting['tokens_roberta'] = df_conversations_greeting['full_conversation'].apply(
    lambda x: count_tokens(x, tokenizer_roberta)
)

# how many exceed the model limits
n_trip_over_longformer = (df_conversations_trip['tokens_longformer'] > 4096).sum()
n_trip_over_roberta = (df_conversations_trip['tokens_roberta'] > 512).sum()

n_greeting_over_longformer = (df_conversations_greeting['tokens_longformer'] > 4096).sum()
n_greeting_over_roberta = (df_conversations_greeting['tokens_roberta'] > 512).sum()

print("Trip > 4096 (Longformer):", n_trip_over_longformer)
print("Trip > 512 (RoBERTa):", n_trip_over_roberta)
print("Greeting > 4096 (Longformer):", n_greeting_over_longformer)
print("Greeting > 512 (RoBERTa):", n_greeting_over_roberta)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1488 > 512). Running this sequence through the model will result in indexing errors


Trip > 4096 (Longformer): 5
Trip > 512 (RoBERTa): 90
Greeting > 4096 (Longformer): 0
Greeting > 512 (RoBERTa): 68


In [23]:
LONGFORMER_MAX = 4096

df_conversations_trip['longformer_excess'] = (
    df_conversations_trip['tokens_longformer'] - LONGFORMER_MAX
).clip(lower=0)

In [24]:
df_conversations_trip.loc[
    df_conversations_trip['longformer_excess'] > 0,
    ['subject_id', 'tokens_longformer', 'longformer_excess', 'subject_group']
].sort_values('longformer_excess', ascending=False)

,subject_id,tokens_longformer,longformer_excess,subject_group
73,665f3e69f83003c1df913675,5217,1121,Young_Adults
88,6772e76feabb30f82fcd8f59,4841,745,Young_Adults
86,673dbbee9983e0d8084c8a76,4606,510,Older_Adults
58,6144d8d264b28a29e10ad3de,4565,469,Young_Adults
84,672107688fa36a5213347a7a,4201,105,Young_Adults


In [25]:
df_model = pd.concat(
    [df_conversations_trip, df_conversations_greeting],
    ignore_index=True
)

In [26]:
df_model['target'] = df_model['subject_group'].map({
    'Young_Adults': 0,
    'Older_Adults': 1
})

X = df_model['full_conversation']
y = df_model['target']

In [27]:
print(df_model.shape)

(180, 11)


In [28]:
df_model

,subject_id,full_conversation,TASK,subject_group,Age,word_count,approx_tokens,tokens_longformer,tokens_roberta,longformer_excess,target
0,568a9909316b10000c50ae46,[user] I would like to spend a four day holida...,trip_task,Older_Adults,75,978,1271.0,1488,1488,0.0,1
1,56b12c0e9f1826000cbcc76c,[user] I want you to help me plan a 4 day trip...,trip_task,Young_Adults,37,1261,1639.0,2031,2031,0.0,0
2,56f699e876348f000c883bba,[user] Please can you help me plan a trip to A...,trip_task,Young_Adults,30,1870,2431.0,2839,2839,0.0,0
3,57698d52688a5c00018d958d,[user] I am trying to plan a 4 day trip to And...,trip_task,Older_Adults,65,2539,3300.0,3930,3930,0.0,1
4,578bb26f6cc44500010448c8,[user] I'd like some help in planning an itine...,trip_task,Older_Adults,70,1694,2202.0,2612,2612,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...
175,6725c4a8e3ab0787551acf38,[user] hi its my daughters ###th birtday what ...,greeting_task,Older_Adults,65,570,NaN,768,768,NaN,1
176,673dbbee9983e0d8084c8a76,[user] Write a birthday greeting for my old sc...,greeting_task,Older_Adults,69,613,NaN,822,822,NaN,1
177,6764260cc5b6436f25820b52,[user] Can you please help me create a birthda...,greeting_task,Young_Adults,31,242,NaN,384,384,NaN,0
178,6772e76feabb30f82fcd8f59,[user] i want a fun birthday card greeting for...,greeting_task,Young_Adults,38,581,NaN,950,950,NaN,0


In [29]:
subjects_df = (
    df_model[['subject_id', 'subject_group', 'target']]
    .drop_duplicates(subset='subject_id')
    .reset_index(drop=True)
)
# Young
young_subjects = subjects_df[subjects_df['target'] == 0]
young_test = young_subjects.sample(n=9, random_state=42)

# Older
older_subjects = subjects_df[subjects_df['target'] == 1]
older_test = older_subjects.sample(n=9, random_state=42)

# combine
test_subjects = pd.concat([young_test, older_test])

train_subjects = subjects_df[~subjects_df['subject_id'].isin(test_subjects['subject_id'])]

In [30]:
test_ids = test_subjects['subject_id']
train_ids = train_subjects['subject_id']

df_test = df_model[df_model['subject_id'].isin(test_ids)].copy()
df_train = df_model[df_model['subject_id'].isin(train_ids)].copy()

In [31]:
print("Test rows:", df_test.shape[0])        #36
print("Train rows:", df_train.shape[0])      #144
print("Test subjects:", df_test['subject_id'].nunique())   # 18
print("Train subjects:", df_train['subject_id'].nunique()) # 72

print("\nTest distribution (subjects):")
print(test_subjects['target'].value_counts())  # 9 & 9

Test rows: 36
Train rows: 144
Test subjects: 18
Train subjects: 72

Test distribution (subjects):
target
0    9
1    9
Name: count, dtype: int64


In [32]:
X_train = df_train['full_conversation'].tolist()
y_train = df_train['target'].values

X_test = df_test['full_conversation'].tolist()
y_test = df_test['target'].values

In [33]:
train_meta = df_train[['subject_id', 'TASK', 'subject_group', 'target']].reset_index(drop=True)
test_meta = df_test[['subject_id', 'TASK', 'subject_group', 'target']].reset_index(drop=True)

In [34]:
from transformers import AutoTokenizer, LongformerModel
import torch

model_name = "allenai/longformer-base-4096"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = LongformerModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LongformerModel(
  (embeddings): LongformerEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(4098, 768, padding_idx=1)
  )
  (encoder): LongformerEncoder(
    (layer): ModuleList(
      (0-11): 12 x LongformerLayer(
        (attention): LongformerAttention(
          (self): LongformerSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (query_global): Linear(in_features=768, out_features=768, bias=True)
            (key_global): Linear(in_features=768, out_features=768, bias=True)
            (value_global): Linear(in_features=768, out_features=768, bias=True)
          )
    

CLS token only (no global attention)

In [35]:
import torch
import numpy as np

def get_embeddings(text_list):
    # Set the model to evaluation mode (disables dropout, etc.)
    model.eval()
    all_embeddings = []

    # Disable gradient calculation to save memory and compute power
    with torch.no_grad():
        for text in text_list:
            # Tokenize the text:
            # - padding: ensures all sequences are the same length
            # - truncation: cuts texts longer than 4096 tokens (solves your outliers)
            # - max_length: the maximum limit for Longformer
            inputs = tokenizer(
                text,
                return_tensors="pt",
                padding="max_length",
                truncation=True,
                max_length=4096
            ).to(device)

            # Forward pass: get the model's internal representation of the text
            outputs = model(**inputs)

            # Extract the [CLS] token embedding:
            # - In Longformer, index 0 is the special [CLS] token that represents the whole document
            # - .cpu().numpy() moves the data from GPU to CPU and converts it to a NumPy array
            cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embedding)

    # Stack all individual embeddings into a single matrix (N_samples, 768)
    return np.vstack(all_embeddings)

# --- Feature Extraction Process ---

# Extract embeddings for the training set
X_train = get_embeddings(df_train['full_conversation'].tolist())
y_train = df_train['target'].values

# Extract embeddings for the test set
X_test = get_embeddings(df_test['full_conversation'].tolist())
y_test = df_test['target'].values

Mean Pooling only (no global attention)

In [36]:
import torch
import numpy as np
from tqdm import tqdm # Import the progress bar library

def get_embeddings(text_list, batch_size=3):
    """
    Extracts embeddings from text using Longformer with Mean Pooling.
    Optimized for CPU/GPU with a progress bar.
    """
    # Set model to evaluation mode
    model.eval()
    all_embeddings = []

    # Use torch.no_grad to disable gradient calculation (saves memory/compute)
    with torch.no_grad():
        # Wrap the range with tqdm to show a progress bar
        # desc: The label shown next to the bar
        # unit: Shows progress in terms of 'batches'
        for i in tqdm(range(0, len(text_list), batch_size), desc="Extracting Embeddings", unit="batch"):
            batch_texts = text_list[i:i + batch_size]

            # Tokenize the current batch
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=4096
            )

            # Move batch to the available device (CPU or GPU)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Get model hidden states
            outputs = model(**inputs)

            # Mean Pooling logic:
            last_hidden = outputs.last_hidden_state
            attention_mask = inputs['attention_mask']

            # Create a mask to ignore padding tokens during averaging
            mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
            masked = last_hidden * mask

            # Calculate sum of embeddings and count non-padding tokens
            summed = masked.sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)

            # Compute the average (Mean Pooling)
            embeddings = summed / counts

            # Store the result as a NumPy array on CPU
            all_embeddings.append(embeddings.cpu().numpy())

    # Combine all batches into a single matrix
    return np.vstack(all_embeddings)

Global Attention + Mean Pooling

In [37]:
def get_embeddings(text_list, batch_size=4):
    model.eval()
    all_embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size), desc="Extracting Embeddings", unit="batch"):
            batch_texts = text_list[i:i + batch_size]

            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=4096
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}

            # NEW: define global attention
            global_attention_mask = torch.zeros_like(inputs["attention_mask"])
            global_attention_mask[:, 0] = 1  # first token gets global attention

            # pass it to the model
            outputs = model(
                **inputs,
                global_attention_mask=global_attention_mask
            )

            # Mean Pooling
            last_hidden = outputs.last_hidden_state
            attention_mask = inputs['attention_mask']

            mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
            masked = last_hidden * mask

            summed = masked.sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)

            embeddings = summed / counts

            all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

Global Attention + CLS token

In [38]:
def get_embeddings(text_list, batch_size=4):
    model.eval()
    all_embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size), desc="Extracting Embeddings", unit="batch"):
            batch_texts = text_list[i:i + batch_size]

            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=4096
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}

            # global attention on first token
            global_attention_mask = torch.zeros_like(inputs["attention_mask"])
            global_attention_mask[:, 0] = 1

            outputs = model(
                **inputs,
                global_attention_mask=global_attention_mask
            )

            # take the first token embedding
            embeddings = outputs.last_hidden_state[:, 0, :]

            all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

In [39]:
y_train = df_train['target'].values
y_test = df_test['target'].values

In [40]:
X_train = get_embeddings(df_train['full_conversation'].tolist())
X_test  = get_embeddings(df_test['full_conversation'].tolist())

Extracting Embeddings: 100%|██████████| 9/9 [00:13<00:00,  1.45s/batch]


In [41]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced')
)

# Train the classifier on your embeddings
clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = clf.predict(X_test)

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# We use a small C value (Regularization) to prevent Overfitting
# Higher C = more complex model, Lower C = simpler model
clf = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced')

# Train the classifier on your embeddings
clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = clf.predict(X_test)

In [43]:
# Print the results
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}")

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.53      0.50      0.51        18
           1       0.53      0.56      0.54        18

    accuracy                           0.53        36
   macro avg       0.53      0.53      0.53        36
weighted avg       0.53      0.53      0.53        36

Accuracy Score: 0.53
